# Phase 5 - Count-aware noise injection (mitigation)
We proved the count is bound to the initial noise and set spatially in the early window. So we mitigate THERE: split the initial latent into N non-overlapping boxes and perturb the in-box noise (Uniform-Scaled / Fixed / Gaussian), then measure exact-count accuracy vs baseline.

**Runtime:** GPU (~20 min).

In [ ]:
import os
if not os.path.exists('src'):
    !git clone https://github.com/serinaqin/T2I-Count-Anomaly.git
    %cd T2I-Count-Anomaly
!pip install -q -r requirements.txt
!pip install -q pytest groundingdino-py

In [ ]:
import sys; sys.path.insert(0, '.')
import numpy as np, pandas as pd, os, yaml, torch
import matplotlib.pyplot as plt
from src.prompts import build_prompt
from src.pipeline import load_sdxl
from src.noise_layout import count_aware_latent
from src.detector import Detector
from src.scoring import count_from_detections, exact_accuracy, mae
from src.config import load_config

In [ ]:
cfg = load_config('configs/phase5.yaml')
raw = yaml.safe_load(open('configs/phase5.yaml'))
schemes = raw['schemes']; obj = cfg.objects[0]
pk = dict(gamma=raw['gamma'], omega=raw['omega'], alpha=raw['alpha'], fill=raw['box_fill'])
pipe = load_sdxl(); det = Detector()
SS = pipe.unet.config.sample_size  # latent H=W (128 for 1024px)
def cnt(img):
    return count_from_detections(det.detect(img, [obj]), obj, cfg.score_threshold)
def base_latent(seed):
    g = torch.Generator(device='cpu').manual_seed(seed)
    z = torch.randn((1, pipe.unet.config.in_channels, SS, SS), generator=g)
    return z.to(pipe.device, pipe.dtype)
def gen_latent(prompt, lat):
    return pipe(prompt, latents=lat, num_inference_steps=cfg.num_inference_steps).images[0]
print('latent', pipe.unet.config.in_channels, 'x', SS, 'x', SS)

In [ ]:
# For each requested N x seed: baseline vs each injection scheme (same base).
rows = []
for N in cfg.counts:
    prompt = build_prompt(N, obj)
    for seed in cfg.seeds:
        base = base_latent(seed)
        rows.append({'N': N, 'seed': seed, 'scheme': 'baseline',
                     'rendered': cnt(gen_latent(prompt, base.clone()))})
        for sch in schemes:
            lat = count_aware_latent(base, N, scheme=sch, **pk)
            rows.append({'N': N, 'seed': seed, 'scheme': sch,
                         'rendered': cnt(gen_latent(prompt, lat))})
    print(f'N={N} done')
df = pd.DataFrame(rows)
df['correct'] = df['rendered'] == df['N']
os.makedirs('results', exist_ok=True)
df.to_csv('results/phase5_counts.csv', index=False)
df.groupby('scheme').agg(exact_acc=('correct', 'mean'),
                         mae=('rendered', lambda s: (df.loc[s.index, 'N'] - s).abs().mean()))

In [ ]:
# Exact-count accuracy: baseline vs schemes, overall and per requested count.
order = ['baseline'] + schemes
acc = df.groupby('scheme')['correct'].mean().reindex(order)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].bar(acc.index, acc.values, color=['gray', 'C0', 'C1', 'C2'])
axes[0].set_ylabel('exact-count accuracy'); axes[0].set_ylim(0, 1)
axes[0].set_title('Overall accuracy vs baseline')
per = df.groupby(['N', 'scheme'])['correct'].mean().unstack().reindex(columns=order)
per.plot(kind='bar', ax=axes[1])
axes[1].set_ylabel('exact-count accuracy'); axes[1].set_title('Accuracy per requested count')
axes[1].legend(fontsize=7)
plt.tight_layout()
plt.savefig('results/phase5_accuracy.png', dpi=100, bbox_inches='tight'); plt.show()
print(acc)

In [ ]:
# Eyeball: baseline vs best scheme for a few (N, seed).
best = acc.drop('baseline').idxmax()
print('best scheme:', best)
picks = [(3, cfg.seeds[0]), (4, cfg.seeds[0]), (5, cfg.seeds[1])]
picks = [(N, s) for (N, s) in picks if N in cfg.counts]
fig, axes = plt.subplots(len(picks), 2, figsize=(6.5, 3.2 * len(picks)))
for row, (N, seed) in enumerate(picks):
    prompt = build_prompt(N, obj); base = base_latent(seed)
    ib = gen_latent(prompt, base.clone())
    ii = gen_latent(prompt, count_aware_latent(base, N, scheme=best, **pk))
    axes[row, 0].imshow(ib); axes[row, 0].axis('off')
    axes[row, 0].set_title(f'asked {N} | baseline={cnt(ib)}', fontsize=9)
    axes[row, 1].imshow(ii); axes[row, 1].axis('off')
    axes[row, 1].set_title(f'asked {N} | {best}={cnt(ii)}', fontsize=9)
plt.tight_layout()
plt.savefig('results/phase5_eyeball.png', dpi=90, bbox_inches='tight'); plt.show()

## How to read this
- **A scheme with clearly higher exact-count accuracy than baseline** = a working, **training-free** mitigation at the level we proved matters (the noise). That closes the arc: interpret -> localize -> prove causal -> characterize (spatial, noise-bound) -> **mitigate**. The eyeball should show the injected images hitting the requested count in coherent scenes, especially at higher counts where baseline collapses.
- **No gain** = the SDXL noise->count coupling needs a stronger layout prior (bigger boxes / perturbation, or the fine-tuning the paper pairs with it, which is out of scope). Report honestly; the causal *finding* stands regardless.
- Watch image quality: over-strong injection can hurt coherence (a quality/accuracy trade-off to note).